# ARK-018 V4 — Science × Birth Book causal campaign

**Use a GPU runtime (T4 is the target) and choose Runtime → Run all.**

This notebook runs the complete preregistered ARK-018 campaign: two initialization seeds × four matched pretraining arms, scientific CONTROL/SEALED evaluation, Birth Book objective probes, narrow algorithmic transfer probes, projected cross-source gradient diagnostics, controlled temporary-binding acquisition/retention, and secondary SciQ scoring when available.

The job is deliberately **resumable across Colab sessions**. Prepared data and checkpoints are stored in Google Drive. If Colab disconnects, reopen this notebook and Run all again; completed preparation/arms are reused and partial arms resume from the last durable checkpoint.

Frozen scientific input path from the operator screenshot:
`/content/drive/MyDrive/genisis-arkenstone/data_15.parquet`

Durable output root:
`/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1/`


In [ ]:
# Reproducible dependency set used by the V4 runner.
%pip -q install 'pyarrow==21.0.0' 'tokenizers==0.21.4' 'datasets==4.0.0'
import torch, pyarrow, tokenizers, datasets
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('pyarrow', pyarrow.__version__, '| tokenizers', tokenizers.__version__, '| datasets', datasets.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime required. In Colab choose Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
SCIENCE = Path('/content/drive/MyDrive/genisis-arkenstone/data_15.parquet')
OUT = Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1')
if not SCIENCE.exists():
    raise FileNotFoundError(f'Expected dataset exactly here: {SCIENCE}')
OUT.mkdir(parents=True, exist_ok=True)
print('SCIENCE:', SCIENCE)
print('Drive-reported local bytes:', SCIENCE.stat().st_size, f'({SCIENCE.stat().st_size/1e6:.1f} MB decimal)')
print('OUTPUT:', OUT)


In [ ]:
# Pin execution to the audited scientific runner commit, not moving branch HEAD.
import os, subprocess, shutil
REPO = Path('/content/An-Ra-the-new-AGI')
PINNED = 'fb0420b7a46521a5f14d125564078ca1c6336d78'
URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git','clone','--no-tags',URL,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','Arkenstone'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',PINNED], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
assert head == PINNED, (head, PINNED)
print('PINNED HEAD:', head)


In [ ]:
# Static compile gate before touching the 1 GB dataset.
import hashlib
FILES = [
    REPO/'experiments/ARK-018/ark018_v3_common.py',
    REPO/'experiments/ARK-018/ark018_v3_binding_fast.py',
    REPO/'experiments/ARK-018/run_ark018_science_birth_v3.py',
    REPO/'experiments/ARK-018/run_ark018_science_birth_v4.py',
]
for p in FILES:
    subprocess.run(['python','-m','py_compile',str(p)], check=True)
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    print('COMPILE PASS', p.name, h)
print('STATIC COMPILE GATE PASS')


## Phase 1 — bind + prepare
This phase streams the **entire Drive Parquet file**, verifies the frozen upstream SHA256 before any model update, validates the Parquet schema, makes content-hash TRAIN/CONTROL/SEALED splits, double-builds the 8,192-token BPE tokenizer for determinism, and writes compact uint16 token caches/checksums to Drive. The Birth Book tokenizer exposure is excluded: vocabulary is learned from scientific TRAIN only.


In [ ]:
RUNNER = REPO/'experiments/ARK-018/run_ark018_science_birth_v4.py'
env = dict(os.environ)
env['PYTHONUNBUFFERED']='1'
env['TOKENIZERS_PARALLELISM']='false'
env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
cmd = ['python', str(RUNNER), '--mode','prepare','--science-path',str(SCIENCE),'--expected-head',PINNED]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, env=env, check=True)
print('PREPARATION / DATA BINDING PASS')


## Phase 2 — CUDA smoke
Preparation already performed the full-file hash gate. The smoke now checks CUDA, Parquet schema, frozen Birth Book identity, source schedules, ~20–25M parameter count, forward/backward, exactly 8,192 targets/update, deterministic same-runtime next-update reproduction, and a Drive checkpoint write/read round trip.


In [ ]:
cmd = ['python', str(RUNNER), '--mode','smoke','--science-path',str(SCIENCE),'--skip-full-hash-smoke','--expected-head',PINNED]
subprocess.run(cmd, cwd=REPO, env=env, check=True)
print('\nGPU SMOKE TEST PASS — GREEN TO START THE FULL CAMPAIGN')


## Phase 3 — full campaign
This executes **all four pretraining arms for both frozen seeds** and all post-training diagnostics. It may span more than one Colab session. Do not reduce arms or seeds just to fit one runtime; rerun the notebook to resume.

Primary causal comparison: `BIRTH_REHEARSAL_10PCT` vs token-matched `SCIENCE_REPLAY_10PCT_CONTROL`. Secondary practical comparison: `BIRTH_NATURAL_2PCT` vs `SCIENCE_ONLY`.


In [ ]:
cmd = ['python', str(RUNNER), '--mode','all','--science-path',str(SCIENCE),'--enable-sciq','--expected-head',PINNED]
proc = subprocess.run(cmd, cwd=REPO, env=env)
print('FULL CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('A failure receipt and every durable completed checkpoint/partial result were left in Drive. Fix the reported blocker, then Run all again; do not delete the output directory.')
    raise RuntimeError(f'ARK-018 runner exited {proc.returncode}')


In [ ]:
# Inspect + download only the compact result archive; model checkpoints remain safely in Drive.
RESULTS = OUT/'results'
print('\nRESULT FILES:')
for p in sorted(RESULTS.glob('*')):
    if p.is_file():
        print(p.name, p.stat().st_size)
zip_path = RESULTS/'ARKENSTONE_ARK018_SCIENCE_BIRTH_RESULTS.zip'
sha_path = RESULTS/'ARKENSTONE_ARK018_SCIENCE_BIRTH_RESULTS.zip.sha256'
if zip_path.exists():
    print('\nFINAL ZIP:', zip_path, zip_path.stat().st_size, 'bytes')
    if sha_path.exists(): print('SHA256:', sha_path.read_text().strip())
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print('Auto-download skipped:', exc)
else:
    print('No final ZIP yet. Re-run all cells to resume the campaign.')
